# Tiny Dreamer Highway — Colab Training Runs

**Name:** Esteban  
**Course:** CSC 580 AI 2  
**Assignment:** Final Project — Dream the Road  
**AI tools consulted:** GitHub Copilot

Use this notebook for actual training experiments, checkpoint generation, and run notes. Keep the setup-and-smoke notebook focused on lightweight validation only.

## Runtime flow

1. Mount Google Drive.
2. Clone or pull the latest validated repository snapshot from GitHub.
3. Install the package from the repo.
4. Set the run name and training overrides.
5. Launch a real training run and save checkpoints/logs to Drive.
6. Inspect the latest metrics summary and checkpoint path.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
REPO_URL = 'https://github.com/estmon8u/CSC_580_Final_Project.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/CSC_580_Final_Project')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'
WORKTREE = Path('/content/CSC_580_Final_Project')

for path in [
    DRIVE_ROOT,
    ARTIFACT_ROOT,
    ARTIFACT_ROOT / 'checkpoints',
    ARTIFACT_ROOT / 'logs',
    ARTIFACT_ROOT / 'training_runs',
]:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)
print('Artifact root:', ARTIFACT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root: /content/drive/MyDrive/CSC_580_Final_Project
Artifact root: /content/drive/MyDrive/CSC_580_Final_Project/artifacts


In [2]:
%%bash
set -e
REPO_URL='https://github.com/estmon8u/CSC_580_Final_Project.git'
if [ ! -d /content/CSC_580_Final_Project/.git ]; then
  git clone "${REPO_URL}" /content/CSC_580_Final_Project
else
  cd /content/CSC_580_Final_Project
  git pull --ff-only origin main
fi

Updating 4eeaef7..cf32343
Fast-forward
 .gitignore                                      |   2 +
 README.md                                       |  13 +
 docs/architecture_overview.md                   |  46 ++-
 docs/colab_git_drive_workflow.md                |   7 +
 docs/evaluation_checklist.md                    |  14 +
 docs/milestones.md                              |  35 ++-
 docs/risk_register.md                           |   1 +
 examples/base_experiment.yaml                   |   4 +
 notebooks/01_colab_setup_and_smoke_tests.ipynb  | 388 +++++++++++++++++++++---
 notebooks/02_colab_training_runs.ipynb          |   0
 notebooks/README.md                             |   8 +-
 pyproject.toml                                  |   1 +
 src/tiny_dreamer_highway/cli.py                 |  96 ++++++
 src/tiny_dreamer_highway/config.py              |   4 +
 src/tiny_dreamer_highway/training/__init__.py   |  10 +
 src/tiny_dreamer_highway/training/experiment.py | 193 ++++++++++++
 src/ti

From https://github.com/estmon8u/CSC_580_Final_Project
 * branch            main       -> FETCH_HEAD
   4eeaef7..cf32343  main       -> origin/main


In [3]:
%%bash
set -e
cd /content/CSC_580_Final_Project
python -m pip install --upgrade pip --quiet
python -m pip install -e . --quiet

In [4]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/CSC_580_Final_Project')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tiny_dreamer_highway.config import load_experiment_config
from tiny_dreamer_highway.training import run_training_experiment

config_path = PROJECT_ROOT / 'examples' / 'base_experiment.yaml'
config = load_experiment_config(config_path)
print('Loaded config from:', config_path)
print('Default cycles:', config.training.cycles)
print('Default warm-start steps:', config.training.warm_start_steps)
print('Default policy steps:', config.training.policy_steps)

Loaded config from: /content/CSC_580_Final_Project/examples/base_experiment.yaml
Default cycles: 10
Default warm-start steps: 64
Default policy steps: 8


## Training run configuration

Adjust the overrides below per experiment. Keep each run in its own Drive-backed subfolder.

In [5]:
RUN_NAME = 'baseline_run_001'
RUN_ARTIFACT_ROOT = ARTIFACT_ROOT / 'training_runs' / RUN_NAME
RUN_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

CYCLES = 25
WARM_START_STEPS = 128
POLICY_STEPS = 8
CHECKPOINT_INTERVAL = 5
RESUME_FROM = None  # example: RUN_ARTIFACT_ROOT / 'checkpoints' / 'checkpoint_00010.pt'

print('Run name:', RUN_NAME)
print('Run artifact root:', RUN_ARTIFACT_ROOT)

Run name: baseline_run_001
Run artifact root: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/training_runs/baseline_run_001


In [6]:
training_summary = run_training_experiment(
    config,
    RUN_ARTIFACT_ROOT,
    cycles=CYCLES,
    warm_start_steps=WARM_START_STEPS,
    policy_steps=POLICY_STEPS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    resume_from=RESUME_FROM,
)

print('Completed cycles:', training_summary.completed_cycles)
print('Replay size:', training_summary.replay_size)
print('Latest checkpoint:', training_summary.latest_checkpoint)
print('Log directory:', training_summary.log_dir)
print('Latest metrics:', training_summary.latest_record)

Completed cycles: 25
Replay size: 328
Latest checkpoint: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/training_runs/baseline_run_001/checkpoints/checkpoint_00025.pt
Log directory: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/training_runs/baseline_run_001/logs
Latest metrics: {'step': 25, 'warm_start_added': 0, 'policy_added': 8, 'replay_size': 328, 'world_model/reconstruction_loss': 0.03616025298833847, 'world_model/reward_loss': 0.010298374108970165, 'world_model/total_loss': 0.04645862802863121, 'behavior/actor_loss': -0.3151538670063019, 'behavior/critic_loss': 0.06625235825777054, 'behavior/imagined_reward_mean': 0.051476914435625076, 'behavior/imagined_value_mean': 0.12570536136627197}


In [7]:
summary_path = training_summary.log_dir / 'latest_summary.json'
summary_payload = json.loads(summary_path.read_text(encoding='utf-8'))
summary_payload

{'checkpoint_file': '/content/drive/MyDrive/CSC_580_Final_Project/artifacts/training_runs/baseline_run_001/checkpoints/checkpoint_00025.pt',
 'latest_metrics': {'behavior/actor_loss': -0.3151538670063019,
  'behavior/critic_loss': 0.06625235825777054,
  'behavior/imagined_reward_mean': 0.051476914435625076,
  'behavior/imagined_value_mean': 0.12570536136627197,
  'policy_added': 8,
  'replay_size': 328,
  'step': 25,
  'warm_start_added': 0,
  'world_model/reconstruction_loss': 0.03616025298833847,
  'world_model/reward_loss': 0.010298374108970165,
  'world_model/total_loss': 0.04645862802863121},
 'latest_step': 25}

## Run notes

Record what changed in this experiment: training length, overrides, qualitative behavior, and whether the checkpoint is worth evaluating further.